# CRM Coursework – Grouping S&P 500 Stocks
Study period: 1 January 2021 – 1 January 2025 (end date exclusive). Benchmark: S&P 500 index (^GSPC). Price data source: yfinance.

In [1]:
# Setup: imports, folders and package versions
import sys, os
import pandas as pd, numpy as np, yfinance as yf, sklearn, scipy, matplotlib

for folder in ['data/raw', 'data/processed', 'outputs/figures']:
    os.makedirs(folder, exist_ok=True)

versions = pd.Series({
    'python': sys.version.split()[0],
    'pandas': pd.__version__,
    'numpy': np.__version__,
    'yfinance': yf.__version__,
    'scikit-learn': sklearn.__version__,
    'scipy': scipy.__version__,
    'matplotlib': matplotlib.__version__,
}, name='version')
versions.to_csv('outputs/package_versions.csv')
versions

python          3.12.0
pandas           3.1.1
numpy            2.5.5
yfinance         1.8.0
scikit-learn     1.9.1
scipy           1.18.1
matplotlib      3.11.2
Name: version, dtype: str

### 1.1 Ticker Selection
I took the S&P 500 company list from Wikipedia, using the version of the page from 20 November 2026 so the list stays the same if the notebook is re-run.

In [2]:
import requests
from io import StringIO
from datetime import datetime

headers = {"User-Agent": "Mozilla/5.0"}

# find the version of the wikipedia page from 20 Nov 2026
params = {
    "action": "query",
    "prop": "revisions",
    "titles": "List of S&P 500 companies",
    "rvlimit": 1,
    "rvstart": "2026-11-20T23:59:59Z",
    "rvdir": "older",
    "format": "json",
}
r = requests.get("https://en.wikipedia.org/w/api.php", params=params, headers=headers)
pages = r.json()["query"]["pages"]
revision_id = list(pages.values())[0]["revisions"][0]["revid"]

url = "https://en.wikipedia.org/w/index.php?title=List_of_S%26P_500_companies&oldid=" + str(revision_id)
html = requests.get(url, headers=headers).text
sp500 = pd.read_html(StringIO(html))[0]

# yfinance needs BRK-B instead of BRK.B
sp500["Ticker"] = sp500["Symbol"].str.replace(".", "-")

sp500["Date added"] = pd.to_datetime(sp500["Date added"], errors="coerce")
sp500["Added after 2021"] = sp500["Date added"] > "2021-01-01"

sp500["Wiki revision"] = revision_id
sp500["Downloaded on"] = datetime.now().strftime("%Y-%m-%d %H:%M")
sp500.to_csv("data/raw/sp500_constituents.csv", index=False)

print("Wikipedia revision:", revision_id)
print("Downloaded on:", sp500["Downloaded on"][0])
print("Number of companies:", len(sp500))
print("Added after 1 Jan 2021:", sp500["Added after 2021"].sum())
sp500.head()

Wikipedia revision: 1384572480
Downloaded on: 2026-11-23 17:36
Number of companies: 503
Added after 1 Jan 2021: 89


,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded,Ticker,Added after 2021,Wiki revision,Downloaded on
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902,MMM,False,1384572480,2026-11-23 17:36
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916,AOS,False,1384572480,2026-11-23 17:36
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888,ABT,False,1384572480,2026-11-23 17:36
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888),ABBV,False,1384572480,2026-11-23 17:36
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989,ACN,False,1384572480,2026-11-23 17:36
